# The full differential geometry toolkit on an arbitrary 2D surface

We consolidate all the geometric notions introduced in this notebook into a single, comprehensive symbolic and numerical exploration on a 2D manifold. While the code below is initialized with the hyperbolic plane in geodesic polar coordinates ($ds^2 = du^2 + \cosh^2(u)\,dv^2$) as a working example, **the entire pipeline is completely metric-agnostic**. You can replace the metric tensor $g_{ij}$ in Part 0 with *any* 2D Riemannian metric, and the toolkit will automatically derive and verify all downstream geometric structures.

We explicitly construct and verify:
1. **Tangent & Cotangent Bundles (Musical Isomorphisms)**: The metric $g$ and its inverse $g^{-1}$ map vector fields (tangent bundle) to 1-forms (cotangent bundle) via the flat ($\flat$) and sharp ($\sharp$) operators.
2. **Levi-Civita Connection**: The Christoffel symbols $\Gamma^k_{ij}$, encoding how basis vectors change. We verify torsion-freeness ($\Gamma^k_{ij} = \Gamma^k_{ji}$) and metric compatibility ($\nabla_k g_{ij} = 0$).
3. **Curvature**: The Riemann tensor $R^\rho_{\sigma\mu\nu}$, its symmetries, the Ricci tensor $R_{\mu\nu}$, and the scalar curvature $R$. We extract the Gauss curvature $K = R/2$.
4. **Differential Forms & Operators**: The exterior derivative $d$, the Hodge star $\star$, the codifferential $\delta$, and the Laplace-Beltrami operator $\Delta = d\delta + \delta d$. We verify $\star^2 = -1$ on 1-forms and that $\Delta$ matches the coordinate formula.
5. **Geodesics & Holonomy**: The geodesic equation $\ddot{x}^k + \Gamma^k_{ij}\dot{x}^i\dot{x}^j = 0$, Jacobi fields (geodesic deviation), and holonomy around closed loops.

This serves as a complete "dictionary" translating abstract differential geometry into the concrete symbolic and numerical machinery of `psiop` and `riemannian`.

In [ ]:
"""
Full Differential Geometry for a Given 2D Metric (Unified Master Edition).

Explores:
  - Setup & Preset Metrics
  - Musical Isomorphisms (♭ and ♯) with Visualizations
  - Levi-Civita Connection, Covariant Derivatives, Parallel Transport
  - Curvature Tensors (Riemann, Ricci, Gauss) & Curvature Maps
  - Differential Forms, Hodge Star, Codifferential, de Rham Laplacian, Pullbacks
  - Vector Calculus Operators (Grad, Div, Curl) & Identities
  - Connection 1-Forms, Curvature 2-Forms, Holonomy, Lie Derivatives & Brackets
  - Geodesics & Jacobi Fields (Geodesic Deviation)
  - Numerical Hodge Decomposition with Visualization
  - Comprehensive Summary & Interaction Diagram
"""

import matplotlib.pyplot as plt
import numpy as np
from psiop import *
from riemannian import *

plt.rcParams.update({"figure.dpi": 100, "font.size": 9})

# ============================================================================
# PART 0: SETUP — The Metric
# ============================================================================
print("=" * 80)
print("PART 0: THE METRIC")
print("=" * 80)

u, v = symbols("u v", real=True)
coords = (u, v)

# Parametrization of surfaces converted into a metric
# Standard Mobius strip parametrization:
#   u in [0, 2*pi)  — goes around the loop
#   v in [-1, 1]     — across the width of the strip
s1 = (1 + v / 2 * cos(u / 2)) * cos(u)
s2 = (1 + v / 2 * cos(u / 2)) * sin(u)
s3 = v / 2 * sin(u / 2)
g_S = surface2metric((s1, s2, s3), (u, v))

# Direct metrics library
R, a = 2, 1  # major / minor radius
g_torus = Matrix([[a**2, 0], [0, (R + a * cos(u)) ** 2]])
g_sphere = Matrix([[1, 0], [0, sin(u) ** 2]])
g_hypbplan = Matrix([[1, 0], [0, cosh(u) ** 2]])
g_paraboloid = Matrix([[1 + u**2, 0], [0, u**2]])
g_warp = Matrix([[1, 0], [0, (1 + u**2) ** 2]])
g_flat = Matrix([[1, 0], [0, 1]])
g_saddle = Matrix([[1, 0], [0, 1 + u**2 + v**2]])
g_zoll = Matrix([[(1 + 0.3 * cos(u)) ** 2, 0], [0, sin(v) ** 2]])
g_clairaut = Matrix([[1, 0], [0, sin(u) ** 2 + 0.5 * cos(u) ** 2]])
g_lens = Matrix([[1, 0], [0, sin(2 * u) ** 2]])

# Selected metric initialized into Metric class object
g_matrix = g_flat
m = Metric(g_matrix, coords)
g_inv = m.g_matrix.inv()
det_g = m.det_g
sqrt_g = m.sqrt_det_g

print(f"\n  Metric g_ij:")
pprint(m.g_matrix)
print(f"\n  Inverse metric g^ij:")
pprint(g_inv)
print(f"  det(g) = {det_g}")
print(f"  √|g|   = {sqrt_g}")

# ============================================================================
# PART 1: MUSICAL ISOMORPHISMS — Vectors ↔ 1-forms (♭ and ♯)
# ============================================================================
print("\n" + "=" * 80)
print("PART 1: MUSICAL ISOMORPHISMS  (♭: TM → T*M,  ♯: T*M → TM)")
print("=" * 80)

V1, V2 = symbols("V1 V2", real=True)
V = (V1, V2)

# Musical isomorphisms using Metric class API
V_flat = m.flat(V)
V_sharp = m.sharp(V_flat)

print(f"\n  V = ({V1}, {V2})  [contravariant vector]")
print(f"  V♭ = g·V = ({V_flat[0]}, {V_flat[1]})  [covariant 1-form]")
print(f"  (V♭)♯ = g⁻¹·V♭ = ({V_sharp[0]}, {V_sharp[1]})")
print(
    f"  Round-trip (V♭)♯ == V: {simplify(V_sharp[0] - V1) == 0 and simplify(V_sharp[1] - V2) == 0}"
)

# --- Visual: how ♭ distorts a vector field ---
print("\n  [VISUAL] Vector field V=(cos v, sin u) and its flat image V♭...")
u_vals = np.linspace(-2, 2, 20)
v_vals = np.linspace(0, 2 * np.pi, 20)
U, V_grid = np.meshgrid(u_vals, v_vals, indexing="ij")

Vx_field = np.cos(V_grid)
Vy_field = np.sin(U)

# Flat: V_flat_u = g_uu * V^u, V_flat_v = g_vv * V^v
Vflat_u = Vx_field
Vflat_v = np.cosh(U) ** 2 * Vy_field

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (fx, fy, title) in zip(
    axes,
    [
        (Vx_field, Vy_field, r"Vector field $V = (\cos v,\, \sin u)$"),
        (Vflat_u, Vflat_v, r"Flat image $V^\flat = g_{ij} V^j$"),
    ],
):
    mag = np.sqrt(fx**2 + fy**2)
    ax.quiver(U, V_grid, fx, fy, mag, cmap="viridis", alpha=0.7, scale=25)
    ax.set_title(title)
    ax.set_xlabel("u")
    ax.set_ylabel("v")
    ax.set_aspect("equal")
plt.tight_layout()
plt.savefig("part1_musical_isomorphisms.png", dpi=150)
plt.show()
print("  → Saved: part1_musical_isomorphisms.png")

# ============================================================================
# PART 2: LEVI-CIVITA CONNECTION — How vectors change along curves
# ============================================================================
print("\n" + "=" * 80)
print("PART 2: LEVI-CIVITA CONNECTION  ∇")
print("=" * 80)

Gamma = m.christoffel_sym
print("\n  Non-zero Christoffel symbols Γ^k_ij:")
for i in range(2):
    for j in range(2):
        for k in range(2):
            if Gamma[i][j][k] != 0:
                print(f"    Γ^{coords[i]}_{coords[j]}{coords[k]} = {Gamma[i][j][k]}")

# Covariant derivatives
Vu, Vv = sin(v), cos(u)
nabla_V = m.covariant_derivative_vector([Vu, Vv])
print(f"\n  Covariant derivative ∇_i V^j:")
print(f"    ∇_u V = ({nabla_V[0,0]}, {nabla_V[0,1]})")
print(f"    ∇_v V = ({nabla_V[1,0]}, {nabla_V[1,1]})")

omega_u, omega_v = cos(u), sin(v)
nabla_omega = m.covariant_derivative_covector([omega_u, omega_v])
print(f"\n  Covariant derivative ∇_i ω_j:")
print(f"    ∇_u ω = ({nabla_omega[0,0]}, {nabla_omega[0,1]})")
print(f"    ∇_v ω = ({nabla_omega[1,0]}, {nabla_omega[1,1]})")

# --- Connection properties: linearity, additivity, Leibniz rule, and more ---
print("\n  --- Properties of the Levi-Civita connection ∇ ---")

Wu, Wv = cos(v), sin(u)
nabla_W = m.covariant_derivative_vector([Wu, Wv])

a_c, b_c = symbols("a_c b_c", real=True)

# Linearity (over constants): ∇(aV + bW) = a∇V + b∇W
combo_field = [a_c * Vu + b_c * Wu, a_c * Vv + b_c * Wv]
nabla_combo = m.covariant_derivative_vector(combo_field)
lin_check = simplify(nabla_combo - (a_c * nabla_V + b_c * nabla_W)) == zeros(2, 2)
print(f"    • Linearity    ∇(aV + bW) = a∇V + b∇W                 : {lin_check}")

# Additivity (special case a = b = 1): ∇(V + W) = ∇V + ∇W
nabla_sum = m.covariant_derivative_vector([Vu + Wu, Vv + Wv])
add_check = simplify(nabla_sum - (nabla_V + nabla_W)) == zeros(2, 2)
print(f"    • Additivity   ∇(V + W) = ∇V + ∇W                     : {add_check}")

# Leibniz / product rule: ∇_i(f V^j) = (∂_i f) V^j + f ∇_i V^j
h_func = Function("h")(u, v)
fV = [h_func * Vu, h_func * Vv]
nabla_fV = m.covariant_derivative_vector(fV)
leibniz_rhs = Matrix(
    [
        [diff(h_func, coords[i]) * [Vu, Vv][j] + h_func * nabla_V[i, j] for j in range(2)]
        for i in range(2)
    ]
)
leibniz_check = simplify(nabla_fV - leibniz_rhs) == zeros(2, 2)
print(f"    • Leibniz      ∇_i(fV^j) = ∂_if·V^j + f∇_iV^j         : {leibniz_check}")

# Same Leibniz rule for covectors: ∇_i(f ω_j) = (∂_i f) ω_j + f ∇_i ω_j
fomega_field = [h_func * omega_u, h_func * omega_v]
nabla_fomega = m.covariant_derivative_covector(fomega_field)
leibniz_cov_rhs = Matrix(
    [
        [diff(h_func, coords[i]) * [omega_u, omega_v][j] + h_func * nabla_omega[i, j] for j in range(2)]
        for i in range(2)
    ]
)
leibniz_cov_check = simplify(nabla_fomega - leibniz_cov_rhs) == zeros(2, 2)
print(f"    • Leibniz      ∇_i(fω_j) = ∂_if·ω_j + f∇_iω_j         : {leibniz_cov_check}")

# Torsion-free: Γ^k_ij = Γ^k_ji  (symmetric in the lower indices — no torsion)
torsion_free = all(
    simplify(Gamma[k][i][j] - Gamma[k][j][i]) == 0
    for k in range(2)
    for i in range(2)
    for j in range(2)
)
print(f"    • Torsion-free Γ^k_ij = Γ^k_ji                        : {torsion_free}")

# Metric compatibility ∇g = 0, written out as ∂_k g_ij = Γ^l_ki g_lj + Γ^l_kj g_il
metric_compat = all(
    simplify(
        diff(m.g_matrix[i, j], coords[k])
        - sum(Gamma[l][k][i] * m.g_matrix[l, j] + Gamma[l][k][j] * m.g_matrix[i, l] for l in range(2))
    )
    == 0
    for k in range(2)
    for i in range(2)
    for j in range(2)
)
print(f"    • Metric comp. ∂_k g_ij = Γ^l_ki g_lj + Γ^l_kj g_il   : {metric_compat}")

# Direct consequence of ∇g = 0 — product rule for the metric inner product:
#   ∂_i⟨V,W⟩ = ⟨∇_i V, W⟩ + ⟨V, ∇_i W⟩
inner_VW = sum(m.g_matrix[j, k] * [Vu, Vv][j] * [Wu, Wv][k] for j in range(2) for k in range(2))
inner_compat_check = all(
    simplify(
        diff(inner_VW, coords[i])
        - sum(
            m.g_matrix[j, k] * nabla_V[i, j] * [Wu, Wv][k] + m.g_matrix[j, k] * [Vu, Vv][j] * nabla_W[i, k]
            for j in range(2)
            for k in range(2)
        )
    )
    == 0
    for i in range(2)
)
print(f"    • Product rule ∂⟨V,W⟩ = ⟨∇V,W⟩ + ⟨V,∇W⟩              : {inner_compat_check}")

# --- Parallel transport along a geodesic (VISUAL) ---
print("\n  [VISUAL] Parallel transport of a vector along a geodesic...")
p0 = (0.5, 0.0)
v0 = (0.3, 1.0)
traj = geodesic_solver(m, p0, v0, (0, 4), method="rk4", n_steps=200)

pt = parallel_transport(m, traj, (1.0, 0.0))

fig, ax = plt.subplots(1, 1, figsize=(10, 6))
ax.plot(traj["x"], traj["y"], "b-", linewidth=2, label="Geodesic")

step = 20
for i in range(0, len(pt["t"]), step):
    x_pos, y_pos = traj["x"][i], traj["y"][i]
    vx, vy = pt["vx"][i], pt["vy"][i]
    scale = 0.3
    ax.arrow(
        x_pos,
        y_pos,
        scale * vx,
        scale * vy,
        head_width=0.05,
        head_length=0.03,
        fc="red",
        ec="red",
        alpha=0.7,
    )
ax.plot(traj["x"][0], traj["y"][0], "go", markersize=10, label="Start")
ax.plot(traj["x"][-1], traj["y"][-1], "rs", markersize=10, label="End")
ax.set_xlabel("u")
ax.set_ylabel("v")
ax.set_title(
    "Parallel Transport\n"
    r"(vector preserves $\langle V, V \rangle_g$ along the geodesic)"
)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig("part2_parallel_transport.png", dpi=150)
plt.show()
print("  → Saved: part2_parallel_transport.png")

# Verify norm preservation
norm_start = (
    m.g_func[(0, 0)](traj["x"][0], traj["y"][0]) * pt["vx"][0] ** 2
    + m.g_func[(1, 1)](traj["x"][0], traj["y"][0]) * pt["vy"][0] ** 2
)
norm_end = (
    m.g_func[(0, 0)](traj["x"][-1], traj["y"][-1]) * pt["vx"][-1] ** 2
    + m.g_func[(1, 1)](traj["x"][-1], traj["y"][-1]) * pt["vy"][-1] ** 2
)
print(
    f"  Norm preservation: |V|²_start = {norm_start:.6f}, |V|²_end = {norm_end:.6f}"
)

# ============================================================================
# PART 3: CURVATURE — Riemann, Ricci, Gauss
# ============================================================================
print("\n" + "=" * 80)
print("PART 3: CURVATURE TENSORS")
print("=" * 80)

R_dict = m.riemann_tensor()
R_down = m.riemann_tensor_lower()

diff_pair = R_down[0][1][0][1] - R_down[1][0][1][0]
diff_anti = R_down[0][1][0][1] + R_down[1][0][0][1]


def is_zero(expr, subs_pt=None):
    e = simplify(expand_trig(expr))
    if e == 0:
        return True
    pt = subs_pt or {u: 0.37, v: 1.1}
    return abs(complex(e.subs(pt))) < 1e-9


symm_check = is_zero(diff_pair)
anti_check1 = is_zero(diff_anti)
print(
    f"\n  Riemann tensor symmetries verified: Pair={symm_check}, Antisym={anti_check1}"
)

Ricci = m.ricci_tensor()
R_scalar = m.ricci_scalar()
K_scalar = m.gauss_curvature()

print(f"\n  Scalar curvature R = {R_scalar}")
print(f"  Gaussian curvature K = {K_scalar}")
print(f"  Ricci Tensor R_ij = K·g_ij:")
pprint(Ricci)

# Trace verification
R_from_trace = m.trace(Ricci, is_covariant=True)
print(
    f"  Verified: Trace(Ricci) == R_scalar ? {simplify(R_from_trace - R_scalar) == 0}"
)

# --- Visual: curvature map ---
print("\n  [VISUAL] Gaussian curvature map...")
fig, ax = plt.subplots(figsize=(8, 6))
u_range = np.linspace(-3, 3, 100)
v_range = np.linspace(0, 2 * np.pi, 100)
U_map, V_map = np.meshgrid(u_range, v_range, indexing="ij")

K_expr = trigsimp(K_scalar)
K_func = lambdify((u, v), K_expr, "numpy")
K_vals = K_func(U_map, V_map)

if np.isscalar(K_vals) or np.shape(K_vals) != np.shape(U_map):
    K_vals = np.full_like(U_map, float(K_vals), dtype=float)

vmin, vmax = np.min(K_vals), np.max(K_vals)
if vmin == vmax:
    vmin, vmax = vmin - 0.5, vmax + 0.5

im = ax.pcolormesh(U_map, V_map, K_vals, cmap="RdBu_r", vmin=vmin, vmax=vmax)
plt.colorbar(im, ax=ax, label="Gaussian Curvature K")
ax.set_xlabel("u")
ax.set_ylabel("v")

if vmin == vmax - 1.0:
    ax.set_title(f"Gaussian Curvature Map\n(Constant $K = {vmin + 0.5:.2f}$)")
else:
    ax.set_title(r"Gaussian Curvature Map (Variable $K$)")

plt.tight_layout()
plt.savefig("part3_curvature_map.png", dpi=150)
plt.show()
print("  → Saved: part3_curvature_map.png")

# Gauss-Bonnet check
gb = verify_gauss_bonnet(m, ((-1, 1), (0, 2 * np.pi)))
print(f"\n  Gauss-Bonnet: ∫∫ K dA = {gb['integral']:.6f}")

# ============================================================================
# PART 4: DIFFERENTIAL FORMS & HODGE THEORY
# ============================================================================
print("\n" + "=" * 80)
print("PART 4: DIFFERENTIAL FORMS & HODGE THEORY")
print("=" * 80)

# Exterior derivative d
f = Function("f")(u, v)
df = (diff(f, u), diff(f, v))
d2f = diff(df[1], u) - diff(df[0], v)
print(f"  df = ({df[0]}) du + ({df[1]}) dv")
print(f"  d²f = ∂_u(∂_v f) - ∂_v(∂_u f) = {d2f}  ✓ (always 0)")

# Hodge Star
star_0 = hodge_star(m, form_degree=0)
star_1 = hodge_star(m, form_degree=1)
star_2 = hodge_star(m, form_degree=2)

omega_u, omega_v = Function("omega_u")(u, v), Function("omega_v")(u, v)
star_omega = star_1(omega_u, omega_v)
star2_omega = star_1(*star_omega)

check_0 = simplify(star2_omega[0] + omega_u) == 0
check_1 = simplify(star2_omega[1] + omega_v) == 0
print(f"  ⋆² on 1-forms: ⋆(⋆ω) = -ω → [{check_0}, {check_1}]  ✓")

# Codifferential δ = -⋆d⋆
alpha_u_expr = sin(u) * cos(v)
alpha_v_expr = cos(u) * sin(v)
delta_alpha = -(1 / sqrt_g) * (
    diff(sqrt_g * (g_inv[0, 0] * alpha_u_expr + g_inv[0, 1] * alpha_v_expr), u)
    + diff(
        sqrt_g * (g_inv[1, 0] * alpha_u_expr + g_inv[1, 1] * alpha_v_expr), v
    )
)
delta_alpha = simplify(delta_alpha)
print(f"  δα for α = sin(u)cos(v) du + cos(u)sin(v) dv:")
print(f"  δα = {delta_alpha}")

# Hodge-de Rham Laplacian Δ = dδ + δd
op0 = de_rham_laplacian(m, form_degree=0)
Delta_f_sym = op0["action"](f)
Delta_f_manual = simplify(
    (1 / sqrt_g)
    * (
        diff(sqrt_g * g_inv[0, 0] * diff(f, u), u)
        + diff(sqrt_g * g_inv[1, 1] * diff(f, v), v)
    )
)
lb_match = simplify(Delta_f_sym - Delta_f_manual) == 0
print(f"  Δ₀f (0-form Laplacian) matches Laplace-Beltrami: {lb_match}")
print(f"  Δ₀f = {Delta_f_manual}")


# Weitzenböck identity
def weitzenbock_gap(m):
    lb = m.laplace_beltrami_symbol()
    K = simplify(m.gauss_curvature())
    D1 = Matrix([[lb["full"] + K, 0], [0, lb["full"] + K]])
    D0 = Matrix([[lb["full"], 0], [0, lb["full"]]])
    gap = simplify(D1 - D0)
    print("\n  Weitzenböck identity check  Δ₁ − ∇*∇ = K·id:")
    print(
        f"  gap == K·id : {simplify(gap - K*Matrix([[1,0],[0,1]])) == Matrix([[0,0],[0,0]])}"
    )
    return gap


weitzenbock_gap(m)

# --- Linearity & Leibniz rule for d and ⋆ ---
print("\n  --- Properties of d and ⋆: linearity & Leibniz rule ---")

g_scalar = Function("g")(u, v)
alpha_u_f, alpha_v_f = Function("alpha_u")(u, v), Function("alpha_v")(u, v)
beta_u_f, beta_v_f = Function("beta_u")(u, v), Function("beta_v")(u, v)
a2, b2 = symbols("a2 b2", real=True)

# Linearity of d on 0-forms: d(af + bg) = a df + b dg
d_combo = (diff(a2 * f + b2 * g_scalar, u), diff(a2 * f + b2 * g_scalar, v))
d_lin_check = (
    simplify(d_combo[0] - (a2 * df[0] + b2 * diff(g_scalar, u))) == 0
    and simplify(d_combo[1] - (a2 * df[1] + b2 * diff(g_scalar, v))) == 0
)
print(f"    • Linearity  d(af + bg) = a df + b dg                 : {d_lin_check}")

# Leibniz rule for d on 0-forms: d(fg) = (df) g + f (dg)
d_fg = (diff(f * g_scalar, u), diff(f * g_scalar, v))
d_fg_rhs = (df[0] * g_scalar + f * diff(g_scalar, u), df[1] * g_scalar + f * diff(g_scalar, v))
leibniz_d0_check = simplify(d_fg[0] - d_fg_rhs[0]) == 0 and simplify(d_fg[1] - d_fg_rhs[1]) == 0
print(f"    • Leibniz    d(fg) = (df)g + f(dg)                    : {leibniz_d0_check}")

# Leibniz rule for d on 1-forms: d(f α) = df ∧ α + f dα
dalpha = diff(alpha_v_f, u) - diff(alpha_u_f, v)
d_falpha = diff(f * alpha_v_f, u) - diff(f * alpha_u_f, v)
df_wedge_alpha = df[0] * alpha_v_f - df[1] * alpha_u_f
leibniz_d1_check = simplify(d_falpha - (df_wedge_alpha + f * dalpha)) == 0
print(f"    • Leibniz    d(fα) = df∧α + f dα                      : {leibniz_d1_check}")

# Linearity of the Hodge star on 1-forms: ⋆(aα + bβ) = a⋆α + b⋆β
star_alpha = star_1(alpha_u_f, alpha_v_f)
star_beta = star_1(beta_u_f, beta_v_f)
combo_1form = (a2 * alpha_u_f + b2 * beta_u_f, a2 * alpha_v_f + b2 * beta_v_f)
star_combo = star_1(*combo_1form)
star_lin_check = (
    simplify(star_combo[0] - (a2 * star_alpha[0] + b2 * star_beta[0])) == 0
    and simplify(star_combo[1] - (a2 * star_alpha[1] + b2 * star_beta[1])) == 0
)
print(f"    • Linearity  ⋆(aα + bβ) = a⋆α + b⋆β                   : {star_lin_check}")

# Pullback demonstration
print("\n  Pullback of a 1-form:")
r, theta = symbols("r theta", real=True, positive=True)
phi_map = (r * cos(theta), r * sin(theta))
omega_uv = (sin(u), cos(v))
omega_rt = m.pullback_1form(phi_map, omega_uv, (r, theta))
print(f"  ω in (u,v) = sin(u) du + cos(v) dv")
print(f"  Pullback to (r,θ) = {omega_rt[0]} dr + {omega_rt[1]} dθ")

print("\n  --- Extended pullback properties: linearity & naturality (φ*d = dφ*) ---")

# Linearity of the pullback: φ*(aω + bη) = a φ*ω + b φ*η
eta_uv = (cos(v), sin(u) * v)
eta_rt = m.pullback_1form(phi_map, eta_uv, (r, theta))
combo_uv = (a2 * omega_uv[0] + b2 * eta_uv[0], a2 * omega_uv[1] + b2 * eta_uv[1])
combo_rt = m.pullback_1form(phi_map, combo_uv, (r, theta))
pullback_lin_check = (
    simplify(combo_rt[0] - (a2 * omega_rt[0] + b2 * eta_rt[0])) == 0
    and simplify(combo_rt[1] - (a2 * omega_rt[1] + b2 * eta_rt[1])) == 0
)
print(f"    • Linearity   φ*(aω + bη) = a φ*ω + b φ*η             : {pullback_lin_check}")

# Naturality: pullback commutes with the exterior derivative, φ*(dh) = d(φ*h)
h_expr = sin(u) * v**2
dh = (diff(h_expr, u), diff(h_expr, v))
dh_pulled = m.pullback_1form(phi_map, dh, (r, theta))
h_pulled = h_expr.subs({u: phi_map[0], v: phi_map[1]})
d_h_pulled = (diff(h_pulled, r), diff(h_pulled, theta))
naturality_check = (
    simplify(dh_pulled[0] - d_h_pulled[0]) == 0 and simplify(dh_pulled[1] - d_h_pulled[1]) == 0
)
print(f"    • Naturality  φ*(dh) = d(φ*h)                         : {naturality_check}")

# ============================================================================
# PART 5: OPERATORS — Gradient, Divergence, Curl, and their interactions
# ============================================================================
print("\n" + "=" * 80)
print("PART 5: VECTOR CALCULUS OPERATORS & INTERACTIONS")
print("=" * 80)

f_example = exp(-(u**2)) * cos(v)
Vu_ex, Vv_ex = sin(v), cos(u) * tanh(u)

grad_f = m.riemannian_gradient(f_example)
div_V = m.divergence((Vu_ex, Vv_ex))
curl_2d = m.curl((Vu_ex, Vv_ex))

print(f"\n  f = exp(-u²)cos(v)")
print(f"  grad f = ({grad_f[0]}, {grad_f[1]})")
print(f"\n  V = (sin(v), cos(u)tanh(u))")
print(f"  div V = {div_V}")
print(f"  curl(V) = {curl_2d}")

print("\n  Fundamental identities:")
div_grad_f = m.divergence(grad_f)
delta_f_direct = op0["action"](f_example)
print(f"    • div(grad f) == Δ₀f : {simplify(div_grad_f - delta_f_direct) == 0}")

curl_grad = m.curl(grad_f)
print(f"    • curl(grad f) == 0  : {simplify(curl_grad) == 0}  ✓")

delta_df = -(1 / sqrt_g) * (
    diff(sqrt_g * (g_inv[0, 0] * diff(f_example, u)), u)
    + diff(sqrt_g * (g_inv[1, 1] * diff(f_example, v)), v)
)
print(f"    • δ(df) == -Δ₀f     : {simplify(delta_df + delta_f_direct) == 0}  ✓")

print("\n  --- Linearity, Leibniz rule & compositions of grad / div / curl ---")

g2_example = sin(u) * v
Wu_ex, Wv_ex = u * cos(v), sin(u) + v

grad_g2 = m.riemannian_gradient(g2_example)
div_W = m.divergence((Wu_ex, Wv_ex))
curl_W = m.curl((Wu_ex, Wv_ex))

a3, b3 = symbols("a3 b3", real=True)

# Linearity of grad: grad(af + bg) = a grad f + b grad g
grad_combo = m.riemannian_gradient(a3 * f_example + b3 * g2_example)
grad_lin_check = (
    simplify(grad_combo[0] - (a3 * grad_f[0] + b3 * grad_g2[0])) == 0
    and simplify(grad_combo[1] - (a3 * grad_f[1] + b3 * grad_g2[1])) == 0
)
print(f"    • Linearity  grad(af+bg) = a grad f + b grad g        : {grad_lin_check}")

# Linearity of div: div(aV + bW) = a div V + b div W
div_combo = m.divergence((a3 * Vu_ex + b3 * Wu_ex, a3 * Vv_ex + b3 * Wv_ex))
div_lin_check = simplify(div_combo - (a3 * div_V + b3 * div_W)) == 0
print(f"    • Linearity  div(aV+bW) = a div V + b div W           : {div_lin_check}")

# Linearity of curl: curl(aV + bW) = a curl V + b curl W
curl_combo = m.curl((a3 * Vu_ex + b3 * Wu_ex, a3 * Vv_ex + b3 * Wv_ex))
curl_lin_check = simplify(curl_combo - (a3 * curl_2d + b3 * curl_W)) == 0
print(f"    • Linearity  curl(aV+bW) = a curl V + b curl W        : {curl_lin_check}")

# Leibniz / product rule for grad: grad(fg) = f grad g + g grad f
grad_prod = m.riemannian_gradient(f_example * g2_example)
grad_prod_rhs = (
    f_example * grad_g2[0] + g2_example * grad_f[0],
    f_example * grad_g2[1] + g2_example * grad_f[1],
)
grad_leibniz_check = (
    simplify(grad_prod[0] - grad_prod_rhs[0]) == 0 and simplify(grad_prod[1] - grad_prod_rhs[1]) == 0
)
print(f"    • Leibniz    grad(fg) = f grad g + g grad f           : {grad_leibniz_check}")

# Leibniz / product rule for div: div(fV) = f div V + ⟨grad f, V⟩_g
inner_gradf_V = sum(m.g_matrix[i, j] * grad_f[i] * [Vu_ex, Vv_ex][j] for i in range(2) for j in range(2))
div_fV = m.divergence((f_example * Vu_ex, f_example * Vv_ex))
div_leibniz_check = simplify(div_fV - (f_example * div_V + inner_gradf_V)) == 0
print(f"    • Leibniz    div(fV) = f div V + ⟨grad f, V⟩_g        : {div_leibniz_check}")

# Composition / Green's 2nd identity: div(f∇g − g∇f) = f Δg − g Δf
Delta_g2 = op0["action"](g2_example)
lhs_green = m.divergence(
    (f_example * grad_g2[0] - g2_example * grad_f[0], f_example * grad_g2[1] - g2_example * grad_f[1])
)
rhs_green = f_example * Delta_g2 - g2_example * delta_f_direct
green_check = simplify(lhs_green - rhs_green) == 0
print(f"    • Composition Green's 2nd id. div(f∇g−g∇f)=fΔg−gΔf   : {green_check}")

# --- Visual: gradient and divergence fields ---
print("\n  [VISUAL] Gradient field and Laplacian of f = exp(-u²)cos(v)...")
u_vis = np.linspace(-2, 2, 25)
v_vis = np.linspace(0, 2 * np.pi, 25)
U_v, V_v = np.meshgrid(u_vis, v_vis, indexing="ij")

f_vals = np.exp(-(U_v**2)) * np.cos(V_v)
grad_u_vals = np.exp(-(U_v**2)) * np.cos(V_v) * (-2 * U_v)
grad_v_vals = np.exp(-(U_v**2)) * (-np.sin(V_v)) / np.cosh(U_v) ** 2

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
mag_grad = np.sqrt(grad_u_vals**2 + grad_v_vals**2)
axes[0].quiver(
    U_v,
    V_v,
    grad_u_vals,
    grad_v_vals,
    mag_grad,
    cmap="plasma",
    alpha=0.8,
    scale=15,
)
axes[0].contour(
    U_v, V_v, f_vals, levels=12, colors="k", alpha=0.3, linewidths=0.5
)
axes[0].set_title(r"Gradient field $\nabla f$ with level sets of $f$")
axes[0].set_xlabel("u")
axes[0].set_ylabel("v")

im = axes[1].pcolormesh(U_v, V_v, f_vals, cmap="RdBu_r", shading="auto")
plt.colorbar(im, ax=axes[1], label="f")
axes[1].set_title(r"Scalar field $f = e^{-u^2}\cos(v)$")
axes[1].set_xlabel("u")
axes[1].set_ylabel("v")
plt.tight_layout()
plt.savefig("part5_gradient_field.png", dpi=150)
plt.show()
print("  → Saved: part5_gradient_field.png")

# ============================================================================
# PART 6: LIE DERIVATIVES, LIE BRACKETS & CONNECTION FORMS
# ============================================================================
print("\n" + "=" * 80)
print("PART 6: LIE DERIVATIVES, LIE BRACKETS & CONNECTION FORMS")
print("=" * 80)

# Lie Brackets & Derivatives
X = (sin(v), cos(u))
Y = (u, v)

bracket = m.lie_bracket(X, Y)
print(f"\n  Lie bracket [X, Y]:")
print(
    f"    [{X[0]}∂_u + {X[1]}∂_v, {Y[0]}∂_u + {Y[1]}∂_v] = {bracket[0]}∂_u + {bracket[1]}∂_v"
)

L_X_g = m.lie_derivative(X, m.g_matrix, obj_type="metric")
is_killing = (
    simplify(L_X_g[0, 0]) == 0
    and simplify(L_X_g[1, 1]) == 0
    and simplify(L_X_g[0, 1]) == 0
)
print(f"\n  Lie derivative of metric along X (Killing check):")
print(f"    L_X g = 0 ? {is_killing}")

omega = (cos(u), sin(v))
L_X_omega = m.lie_derivative(X, omega, obj_type="1form")
print(f"\n  Lie derivative of 1-form ω = (cos(u), sin(v)) along X:")
print(f"    L_X ω = ({L_X_omega[0]}, {L_X_omega[1]})")

# --- Properties of Lie brackets & Lie derivatives ---
print("\n  --- Properties of Lie brackets & Lie derivatives ---")

Z = (u * v, u - v)  # third vector field, needed for the Jacobi identity

# Antisymmetry of the Lie bracket: [X, Y] = -[Y, X]
bracket_YX = m.lie_bracket(Y, X)
antisym_check = simplify(bracket[0] + bracket_YX[0]) == 0 and simplify(bracket[1] + bracket_YX[1]) == 0
print(f"    • Antisymmetry [X,Y] = -[Y,X]                         : {antisym_check}")

# Bilinearity (additivity + linearity over constants): [aX+bY, Z] = a[X,Z] + b[Y,Z]
a4, b4 = symbols("a4 b4", real=True)
combo_XY = (a4 * X[0] + b4 * Y[0], a4 * X[1] + b4 * Y[1])
bracket_combo_Z = m.lie_bracket(combo_XY, Z)
bracket_X_Z = m.lie_bracket(X, Z)
bracket_Y_Z = m.lie_bracket(Y, Z)
bilin_check = (
    simplify(bracket_combo_Z[0] - (a4 * bracket_X_Z[0] + b4 * bracket_Y_Z[0])) == 0
    and simplify(bracket_combo_Z[1] - (a4 * bracket_X_Z[1] + b4 * bracket_Y_Z[1])) == 0
)
print(f"    • Bilinearity  [aX+bY, Z] = a[X,Z] + b[Y,Z]           : {bilin_check}")

# Leibniz rule for the bracket: [X, fY] = f[X,Y] + (Xf) Y
h_lb = exp(u) * sin(v)
fY = (h_lb * Y[0], h_lb * Y[1])
bracket_X_fY = m.lie_bracket(X, fY)
Xf = X[0] * diff(h_lb, u) + X[1] * diff(h_lb, v)
rhs_leibniz_bracket = (h_lb * bracket[0] + Xf * Y[0], h_lb * bracket[1] + Xf * Y[1])
leibniz_bracket_check = (
    simplify(bracket_X_fY[0] - rhs_leibniz_bracket[0]) == 0
    and simplify(bracket_X_fY[1] - rhs_leibniz_bracket[1]) == 0
)
print(f"    • Leibniz      [X,fY] = f[X,Y] + (Xf)Y                : {leibniz_bracket_check}")

# Jacobi identity: [[X,Y],Z] + [[Y,Z],X] + [[Z,X],Y] = 0
bracket_YZ = m.lie_bracket(Y, Z)
bracket_ZX = m.lie_bracket(Z, X)
jac1 = m.lie_bracket(bracket, Z)
jac2 = m.lie_bracket(bracket_YZ, X)
jac3 = m.lie_bracket(bracket_ZX, Y)
jacobi_check = (
    simplify(jac1[0] + jac2[0] + jac3[0]) == 0 and simplify(jac1[1] + jac2[1] + jac3[1]) == 0
)
print(f"    • Jacobi id.   [[X,Y],Z]+[[Y,Z],X]+[[Z,X],Y] = 0      : {jacobi_check}")

# Linearity of the Lie derivative (over constants), tested on the metric
L_Y_g = m.lie_derivative(Y, m.g_matrix, obj_type="metric")
L_combo_g = m.lie_derivative(combo_XY, m.g_matrix, obj_type="metric")
lie_lin_check = simplify(L_combo_g - (a4 * L_X_g + b4 * L_Y_g)) == zeros(2, 2)
print(f"    • Linearity    L_(aX+bY) g = a L_X g + b L_Y g        : {lie_lin_check}")

# Leibniz rule for the Lie derivative on a 1-form: L_X(fω) = (Xf) ω + f L_X ω
f_lb = cos(u) * v
fomega = (f_lb * omega[0], f_lb * omega[1])
L_X_fomega = m.lie_derivative(X, fomega, obj_type="1form")
Xf_lb = X[0] * diff(f_lb, u) + X[1] * diff(f_lb, v)
rhs_lie_leibniz = (Xf_lb * omega[0] + f_lb * L_X_omega[0], Xf_lb * omega[1] + f_lb * L_X_omega[1])
lie_leibniz_check = (
    simplify(L_X_fomega[0] - rhs_lie_leibniz[0]) == 0
    and simplify(L_X_fomega[1] - rhs_lie_leibniz[1]) == 0
)
print(f"    • Leibniz      L_X(fω) = (Xf)ω + f L_X ω              : {lie_leibniz_check}")

# Naturality: Lie derivatives represent the Lie bracket, [L_X, L_Y] = L_[X,Y]
LY_omega = m.lie_derivative(Y, omega, obj_type="1form")
LX_LY_omega = m.lie_derivative(X, LY_omega, obj_type="1form")
LY_LX_omega = m.lie_derivative(Y, L_X_omega, obj_type="1form")
L_bracket_omega = m.lie_derivative(bracket, omega, obj_type="1form")
naturality_lie_check = (
    simplify(LX_LY_omega[0] - LY_LX_omega[0] - L_bracket_omega[0]) == 0
    and simplify(LX_LY_omega[1] - LY_LX_omega[1] - L_bracket_omega[1]) == 0
)
print(f"    • Naturality   L_X L_Y ω − L_Y L_X ω = L_[X,Y] ω      : {naturality_lie_check}")

# Connection 1-form & Curvature 2-form
E = simplify(m.g_matrix[0, 0])
G = simplify(m.g_matrix[1, 1])
sqrt_EG = simplify(sqrt(E * G))

if m.g_matrix[0, 1] != 0 or m.g_matrix[1, 0] != 0:
    print("\n  [WARNING] Metric contains off-diagonal terms.")

A = simplify(diff(E, v) / (2 * sqrt_EG))
B = simplify(-diff(G, u) / (2 * sqrt_EG))

omega_12_u, omega_12_v = simplify(A), simplify(B)
print(f"\n  Connection 1-form: ω¹₂ = ({omega_12_u}) du + ({omega_12_v}) dv")

Omega_12_coeff = simplify(diff(B, u) - diff(A, v))
K_from_Omega = simplify(Omega_12_coeff / sqrt_EG)
print(f"  Curvature 2-form: Ω¹₂ = dω¹₂ = ({Omega_12_coeff}) du∧dv")
print(
    f"  Gauss Curvature via Ω¹₂: K = {K_from_Omega} (Matches: {is_zero(K_from_Omega - K_scalar)})"
)

# --- Holonomy Visualization ---
print("\n  [VISUAL] Holonomy: parallel transport around a rectangular loop...")
eps, u0, v0 = 0.3, 0.5, 0.5
t_total, n_pts = 1.0, 400
t_path = np.linspace(0, t_total, n_pts)
seg = n_pts // 4

# Fixed loop coordinates returning to (u0, v0)
path_u = np.concatenate(
    [
        np.linspace(u0, u0 + eps, seg),
        np.full(seg, u0 + eps),
        np.linspace(u0 + eps, u0, seg),
        np.full(seg, u0),
    ]
)
path_v = np.concatenate(
    [
        np.full(seg, v0),
        np.linspace(v0, v0 + eps, seg),
        np.full(seg, v0 + eps),
        np.linspace(v0 + eps, v0, seg),  # FIXED: goes from v0 + eps down to v0
    ]
)

curve_dict = {"t": t_path, "x": path_u, "y": path_v}
pt_loop = parallel_transport(m, curve_dict, (1.0, 0.0))

v_start = np.array([pt_loop["vx"][0], pt_loop["vy"][0]])
v_end = np.array([pt_loop["vx"][-1], pt_loop["vy"][-1]])

# Retrieve metric components at the starting/ending point
g00 = m.g_func[(0, 0)](path_u[0], path_v[0])
g01 = m.g_func[(0, 1)](path_u[0], path_v[0])
g11 = m.g_func[(1, 1)](path_u[0], path_v[0])

# Metric inner product and norms: <u, v>_g = u^T G v
norm_s = np.sqrt(
    g00 * v_start[0] ** 2 + 2 * g01 * v_start[0] * v_start[1] + g11 * v_start[1] ** 2
)
norm_e = np.sqrt(
    g00 * v_end[0] ** 2 + 2 * g01 * v_end[0] * v_end[1] + g11 * v_end[1] ** 2
)

g_inner_prod = (
    g00 * v_start[0] * v_end[0]
    + g01 * (v_start[0] * v_end[1] + v_start[1] * v_end[0])
    + g11 * v_start[1] * v_end[1]
)

cos_angle = g_inner_prod / (norm_s * norm_e)
holonomy_angle = np.arccos(np.clip(cos_angle, -1.0, 1.0))

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(path_u, path_v, "b-", linewidth=2)
ax.plot(path_u[0], path_v[0], "go", markersize=12, label="Start/End")

scale = 0.15
ax.arrow(
    path_u[0],
    path_v[0],
    scale * v_start[0] / norm_s,
    scale * v_start[1] / norm_s,
    head_width=0.02,
    fc="green",
    ec="green",
    linewidth=2,
    label="V (start)",
)
ax.arrow(
    path_u[-1],
    path_v[-1],
    scale * v_end[0] / norm_e,
    scale * v_end[1] / norm_e,
    head_width=0.02,
    fc="red",
    ec="red",
    linewidth=2,
    label="V (after loop)",
)
ax.set_xlabel("u")
ax.set_ylabel("v")
ax.set_title(
    f"Holonomy \nVector rotates by ≈ {holonomy_angle:.3f} rad after loop"
)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig("part6_holonomy.png", dpi=150)
plt.show()
print("  → Saved: part6_holonomy.png")

# ============================================================================
# PART 7: GEODESICS & JACOBI FIELDS
# ============================================================================
print("\n" + "=" * 80)
print("PART 7: GEODESICS & JACOBI FIELDS (Geodesic Deviation)")
print("=" * 80)

# Symbolic geodesic equations
t = symbols("t", real=True)
u_t, v_t = Function("u")(t), Function("v")(t)
du_dt, dv_dt = diff(u_t, t), diff(v_t, t)

geo_u = diff(u_t, t, 2) + sum(
    Gamma[0][i][j] * [du_dt, dv_dt][i] * [du_dt, dv_dt][j]
    for i in range(2)
    for j in range(2)
)
geo_v = diff(v_t, t, 2) + sum(
    Gamma[1][i][j] * [du_dt, dv_dt][i] * [du_dt, dv_dt][j]
    for i in range(2)
    for j in range(2)
)

print(f"\n  Geodesic equations:")
print(f"    u'': {simplify(geo_u)} = 0")
print(f"    v'': {simplify(geo_v)} = 0")

p0 = (np.pi / 2, 0.0)
K_start = float(
    trigsimp(m.gauss_curvature()).subs(dict(zip(m.coords, p0)))
)

if K_start < -1e-12:
    geo_title = rf"Geodesic Family ($K = {K_start:.2f} < 0$: exponential divergence)"
    jac_title = rf"Jacobi Field ($K = {K_start:.2f}$: Exponential growth)"
elif K_start > 1e-12:
    geo_title = (
        rf"Geodesic Family ($K = {K_start:.2f} > 0$: convergence at conjugate points)"
    )
    jac_title = (
        rf"Jacobi Field ($K = {K_start:.2f}$: Oscillatory $\sim \sin(\sqrt{{K}}\,t)$)"
    )
else:
    geo_title = r"Geodesic Family ($K = 0$: linear divergence)"
    jac_title = r"Jacobi Field ($K = 0$: Linear growth)"

# Geodesic family plot
print("\n  [VISUAL] Geodesic family ...")
fig, ax = plt.subplots(figsize=(10, 7))

n_geod = 7
angles = np.linspace(-0.45, 0.45, n_geod)
colors = plt.cm.viridis(np.linspace(0, 1, n_geod))

for angle, color in zip(angles, colors):
    v0_geod = (np.sin(angle), np.cos(angle))
    traj = geodesic_solver(
        m, p0, v0_geod, (0, 3.2), method="rk4", n_steps=400
    )
    ax.plot(traj["x"], traj["y"], color=color, linewidth=1.5, alpha=0.8)
    ax.plot(traj["x"][0], traj["y"][0], "o", color=color, markersize=4)

ax.set_xlabel(m.coords[0].name)
ax.set_ylabel(m.coords[1].name)
ax.set_title(geo_title)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Jacobi field solver & plot
ref_geod = geodesic_solver(
    m, p0, (0.0, 1.0), (0, 3.5), method="rk4", n_steps=400
)
jac = jacobi_equation_solver(
    m,
    ref_geod,
    {"J0": (0.1, 0.0), "DJ0": (0.0, 0.0)},
    (0, 3.5),
    n_steps=400,
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(jac["t"], jac["J_x"], "b-", label=rf"$J^{{{m.coords[0]}}}(t)$")
ax.plot(jac["t"], jac["J_y"], "r-", label=rf"$J^{{{m.coords[1]}}}(t)$")

x_on_jac_t = np.interp(jac["t"], ref_geod["t"], ref_geod["x"])
y_on_jac_t = np.interp(jac["t"], ref_geod["t"], ref_geod["y"])

g00 = m.g_func[(0, 0)](x_on_jac_t, y_on_jac_t)
g01 = m.g_func[(0, 1)](x_on_jac_t, y_on_jac_t)
g11 = m.g_func[(1, 1)](x_on_jac_t, y_on_jac_t)

norm_J = np.sqrt(
    np.maximum(
        g00 * jac["J_x"] ** 2
        + 2 * g01 * jac["J_x"] * jac["J_y"]
        + g11 * jac["J_y"] ** 2,
        0.0,
    )
)

ax.plot(jac["t"], norm_J, "k--", linewidth=2, label=r"$\|J(t)\|_g$")

if K_start > 1e-12:
    t_conj = np.pi / np.sqrt(K_start)
    ax.axvline(
        t_conj,
        color="gray",
        linestyle=":",
        label=rf"Conjugate point ($t = \pi/\sqrt{{K}} \approx {t_conj:.3f}$)",
    )

ax.set_xlabel("t (arc length)")
ax.set_ylabel("Jacobi field components")
ax.set_title(jac_title)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================================
# PART 8: NUMERICAL HODGE DECOMPOSITION
# ============================================================================
print("\n" + "=" * 80)
print("PART 8: NUMERICAL HODGE DECOMPOSITION  α = dφ + ⋆dψ + h")
print("=" * 80)

alpha_x_expr = sin(v) * cosh(u)
alpha_y_expr = cos(u) * cos(v)
domain = ((-1, 1), (0, 2 * np.pi))

print(f"\n  Decomposing α = {alpha_x_expr} d{u} + {alpha_y_expr} d{v}")
decomp = hodge_decomposition(
    m, (alpha_x_expr, alpha_y_expr), domain, resolution=70, form_degree=1
)

print(f"\n  Decomposition complete!")
print(f"  Exact part energy:     {np.sum(decomp['alpha_exact']**2):.4f}")
print(f"  Co-exact part energy:  {np.sum(decomp['alpha_coexact']**2):.4f}")
print(f"  Harmonic part energy:  {np.sum(decomp['alpha_harmonic']**2):.4f}")

analyze_hodge_decomposition(
    decomp,
    original=(alpha_x_expr, alpha_y_expr),
    print_report=True,
    show_plot=True,
)

# ============================================================================
# PART 9: SUMMARY — The Full Interaction Diagram
# ============================================================================
print("\n" + "=" * 80)
print("PART 9: SUMMARY — HOW EVERYTHING CONNECTS")
print("=" * 80)


def print_summary(m, title=None):
    x, y = m.coords
    K = simplify(m.gauss_curvature())
    R = simplify(m.ricci_scalar())
    Ric = m.ricci_tensor().applyfunc(simplify)
    g = m.g_matrix

    rule = "-" * 72
    print(rule)
    print(title or f"Metric summary on ({x}, {y})")
    print(rule)
    print(f"  g   = [[{g[0,0]}, {g[0,1]}], [{g[1,0]}, {g[1,1]}]]")
    print(f"  K   = {K}")
    print(f"  R   = {R}")
    print(f"  Ric = [[{Ric[0,0]}, {Ric[0,1]}], [{Ric[1,0]}, {Ric[1,1]}]]")

    print(rule)
    print("Musical / Hodge square")
    print(rule)
    print("  0-form f  --d-->  1-form df          vector V")
    print("      |⋆                 |⋆                |♭")
    print("      v                  v                 v")
    print("  2-form *f <--d--  1-form *df         1-form V^flat = g.V")

    print(rule)
    print("Connection ∇ acts on")
    print(rule)
    print("  vectors:  ∇_i V^j = ∂_i V^j + Γ^j_ik V^k")
    print("  1-forms:  ∇_i ω_j = ∂_i ω_j - Γ^k_ij ω_k")
    print("  preserves the metric:  ∇g = 0")

    print(rule)
    print("Curvature controls")
    print(rule)
    print("  geodesic deviation:  J'' + K.J = 0  ->  J ~ exp(sqrt(-K).t)")
    print("  holonomy:            angle = ∫∫ K dA   (exact, any simple loop)")
    print("  Weitzenböck:         Δ_1 = ∇*∇ + K.id")

    print(rule)
    print("Hodge theory & Lie Structure")
    print(rule)
    print("  d² = 0                        (topological, metric-independent)")
    print("  δ = -⋆d⋆                      (metric-dependent adjoint)")
    print("  Δ = dδ + δd                   (Hodge-de Rham Laplacian)")
    print("  ⋆² = (-1)^(k(n-k))            on k-forms in n dimensions")
    print("  hodge decomposition:  α = dφ + ⋆dψ + h")
    print("  [X, Y]                        (Lie bracket commutator)")
    print("  L_X g = 0                     (Killing vector field condition)")

    print(rule)
    print("Key operator identities")
    print(rule)
    print("  div(grad f) = Δ_0 f")
    print("  curl(grad f) = 0             (= d²f = 0)")
    print("  δ(df) = -Δ_0 f")
    print("  ⋆d⋆ = -δ                     (codifferential via Hodge star)")
    print("  Ω¹₂ = K.dA                   (curvature 2-form = K x area form)")
    print(rule)


print_summary(m, title="Master Metric Summary")

print("\n→ Example complete!")
print("  All geometric structures verified symbolically and numerically.")

In [ ]:
# ============================================================================
# bridge_psiop.py — Realize every riemannian.Metric operator as a ψDO
# ----------------------------------------------------------------------------
# The message this module makes explicit:
#   • The metric  g  lives in the PRINCIPAL symbol (g^ij ξ_i ξ_j).
#   • Curvature   K  lives in the LOWER-ORDER / subprincipal terms.
#   • Geodesics are the bicharacteristics of the principal symbol.
# ============================================================================

# ---------------------------------------------------------------------------
# Cotangent variables (same convention used by both modules: xi, eta)
# ---------------------------------------------------------------------------
def _freq(m):
    """Cotangent symbols matching psiop's internal convention."""
    if m.dim == 1:
        return (symbols('xi', real=True),)
    return symbols('xi eta', real=True)


# ---------------------------------------------------------------------------
# Dirac / signature operator  D = d + δ : Ω^even -> Ω^odd
# In 2D, Ω^even = Ω^0 ⊕ Ω^2 has 2 scalar components (f, h) and Ω^1 has 2
# components, so D is a genuine 2×2 matrix operator with D² = Δ.
# ---------------------------------------------------------------------------
def _dirac_symbol_matrix(m):
    """
    Build the 2×2 symbol matrix of D = d + δ acting on (f, h):
        column 1 : d on the 0-form f
        column 2 : δ on the 2-form h·du∧dv   (δ = -⋆d⋆)
    The Hodge star is built from the INVERSE metric so that ⋆² = -id.
    """
    x, y = m.coords
    xi, eta = symbols('xi eta', real=True)
    sqrt_g = m.sqrt_det_g
    ginv   = m.g_inv_matrix

    theta = I * (x * xi + y * eta)
    E     = exp(theta)                       # plane wave test input

    # ---- column 1 : d f,  f = E  ->  symbol (iξ, iη) ----
    c1u = simplify(diff(E, x) / E)
    c1v = simplify(diff(E, y) / E)

    # ---- column 2 : δ(h du∧dv),  h = E,  δ = -⋆d⋆ ----
    phi    = E / sqrt_g                       # ⋆(h du∧dv)
    dphi_u = diff(phi, x)
    dphi_v = diff(phi, y)
    # ⋆ of the 1-form (dphi_u, dphi_v), inverse-metric convention
    star_u =  sqrt_g * (ginv[1, 0] * dphi_u + ginv[1, 1] * dphi_v)
    star_v = -sqrt_g * (ginv[0, 0] * dphi_u + ginv[0, 1] * dphi_v)
    c2u = simplify(-star_u / E)               # δ = -⋆d⋆
    c2v = simplify(-star_v / E)

    return Matrix([[c1u, c2u],
                   [c1v, c2v]])


# ---------------------------------------------------------------------------
# THE REGISTRY — every usable operator as a real ψDO object
# ---------------------------------------------------------------------------
def metric_symbol_registry(m, quantization='kohn-nirenberg',
                           apply_backend='peetre'):
    """
    Return {name: PseudoDifferentialOperator | MatrixPseudoDifferentialOperator}
    for every geometric operator of the metric that is a genuine (square)
    endomorphism on sections.

    Scalar operators:
        laplace_beltrami      Δ₀ = |g|^-½ ∂ᵢ(|g|^½ g^ij ∂ⱼ)      order 2
        gaussian_curvature    multiplication by K                  order 0
        ricci_scalar          multiplication by R                  order 0
    Matrix operators (2D):
        derham_laplacian_1form   Δ₁ = ∇*∇ + K·id   (Weitzenböck)   order 2
        rough_laplacian_1form    ∇*∇               (no curvature)  order 2
        dirac_operator           D = d + δ,  D² = Δ                order 1
    """
    kw = dict(mode='symbol', quantization=quantization,
              apply_backend=apply_backend)
    reg = {}

    # ---- scalar Laplace–Beltrami Δ₀ (principal + subprincipal) ----
    lb = m.laplace_beltrami_symbol()
    reg['laplace_beltrami'] = PseudoDifferentialOperator(
        lb['full'], list(m.coords), **kw)

    # ---- order-0 multiplication operators (curvature as symbol) ----
    reg['gaussian_curvature'] = PseudoDifferentialOperator(
        simplify(m.gauss_curvature()), list(m.coords), **kw)
    reg['ricci_scalar'] = PseudoDifferentialOperator(
        simplify(m.ricci_scalar()), list(m.coords), **kw)

    if m.dim == 2:
        K  = simplify(m.gauss_curvature())
        D1 = lb['full'] + K                       # Δ₁ diagonal block
        D0 = lb['full']                            # ∇*∇ diagonal block

        reg['derham_laplacian_1form'] = MatrixPseudoDifferentialOperator(
            Matrix([[D1, 0], [0, D1]]), list(m.coords), **kw)
        reg['rough_laplacian_1form'] = MatrixPseudoDifferentialOperator(
            Matrix([[D0, 0], [0, D0]]), list(m.coords), **kw)
        reg['dirac_operator'] = MatrixPseudoDifferentialOperator(
            _dirac_symbol_matrix(m), list(m.coords), **kw)

    return reg


# ---------------------------------------------------------------------------
# Display metadata for the FULL table (incl. first-order / non-square ops)
# ---------------------------------------------------------------------------
def _symbol_table_rows(m, X_field=None):
    """List of dicts {name, normal, symbol, order, kind} for pretty-printing."""
    x, y = m.coords
    xi, eta = _freq(m)
    ginv   = m.g_inv_matrix
    g      = m.g_matrix
    sqrt_g = m.sqrt_det_g
    lb     = m.laplace_beltrami_symbol()
    K      = simplify(m.gauss_curvature())
    rows   = []

    def add(name, normal, symbol, order, kind):
        rows.append({'name': name, 'normal': normal, 'symbol': symbol,
                     'order': order, 'kind': kind})

    # ======================================================================
    # NEW: MUSICAL ISOMORPHISMS (Order 0 Bundle Isomorphisms)
    # ======================================================================
    add('musical flat ♭',
        'V_i = g_ij V^j',
        g, 0, 'vector → 1-form')
    
    add('musical sharp ♯',
        'ω^i = g^ij ω_j',
        ginv, 0, '1-form → vector')

    # ======================================================================
    # First-order operators
    # ======================================================================
    add('exterior derivative d',
        'df = (∂ⱼf) dxʲ',
        Matrix([I*xi, I*eta]),
        1, '0-form → 1-form')
        
    add('gradient grad',
        '(grad f)ⁱ = gⁱʲ ∂ⱼf',
        Matrix([I*(ginv[0,0]*xi + ginv[0,1]*eta),
                I*(ginv[1,0]*xi + ginv[1,1]*eta)]),
        1, '0-form → vector')
        
    add('divergence div',
        'div(V) = (1/√g) ∂ᵢ(√g Vⁱ)',
        Matrix([[I*xi, I*eta]]), # Principal symbol is i ξ_j contracted with V^j
        1, 'vector → 0-form')
        
    add('codifferential δ',
        'δα = -(1/√g) ∂ᵢ(√g gⁱʲ αⱼ)',
        Matrix([[-I*(ginv[0,0]*xi + ginv[0,1]*eta),
                 -I*(ginv[1,0]*xi + ginv[1,1]*eta)]]),
        1, '1-form → 0-form')
        
    add('curl (2D) = ⋆d',
        '(1/√g)(∂ᵤαᵥ - ∂ᵥαᵤ)',
        Matrix([[-I*eta/sqrt_g, I*xi/sqrt_g]]),
        1, '1-form → 0-form')

    # ======================================================================
    # NEW: LIE DERIVATIVE (Order 1, depends on a fixed vector field X)
    # ======================================================================
    if X_field is not None and m.dim == 2:
        X1, X2 = X_field
        # Symbol of L_X on scalars is i X^j \xi_j
        lie_sym = I * (X1 * xi + X2 * eta)
        add('Lie derivative L_X',
            f'L_X f = X^j ∂_j f  (with X=({X1}, {X2}))',
            lie_sym, 1, '0-form → 0-form')
        # Note: If {g^ij \xi_i \xi_j, X^k \xi_k} = 0, X is a Killing field!

    # ======================================================================
    # Zero-order operators
    # ======================================================================
    star_mat = Matrix([[-sqrt_g*ginv[1,0], -sqrt_g*ginv[1,1]],
                       [ sqrt_g*ginv[0,0],  sqrt_g*ginv[0,1]]])
    add('Hodge star ⋆',
        'fiber isomorphism (order 0)',
        star_mat, 0, '1-form → 1-form')
    add('Gaussian curvature K',
        'multiplication by K', K, 0, '0-form → 0-form')
    add('Ricci scalar R',
        'multiplication by R', simplify(m.ricci_scalar()), 0, '0-form → 0-form')

    # ======================================================================
    # Second-order operators
    # ======================================================================
    add('Laplace–Beltrami Δ₀',
        '|g|^-½ ∂ᵢ(|g|^½ gⁱʲ ∂ⱼ)',
        lb['full'], 2, '0-form → 0-form')
    if m.dim == 2:
        add('de Rham Laplacian Δ₁',
            '∇*∇ + K·id  (Weitzenböck)',
            Matrix([[lb['full']+K, 0], [0, lb['full']+K]]),
            2, '1-form → 1-form')
        add('rough Laplacian ∇*∇',
            'component-wise scalar Laplacian',
            Matrix([[lb['full'], 0], [0, lb['full']]]),
            2, '1-form → 1-form')
        add('Dirac operator d+δ',
            'D = d + δ ,  D² = Δ',
            _dirac_symbol_matrix(m), 1, 'Ω^even → Ω^odd')
            
    return rows


# ---------------------------------------------------------------------------
# Pretty-print the table
# ---------------------------------------------------------------------------
def print_symbol_table(m, X_field=None, use_latex=False):
    """
    Pretty-print:  name | normal form | ψDO symbol | order | kind
    """
    rows = _symbol_table_rows(m, X_field=X_field)
    line = '─' * 78
    print(f"\nMetric g on {tuple(m.coords)}   (dim = {m.dim})")
    print(f"√|g| = {m.sqrt_det_g}\n")
    for r in rows:
        print(line)
        print(f"  {r['name']}    [{r['kind']}]    order {r['order']}")
        print(f"    normal : {r['normal']}")
        sym = r['symbol']
        if use_latex:
            print(f"    symbol : ${latex(sym)}$")
        else:
            print(f"    symbol : {sym}")
    print(line)


# ---------------------------------------------------------------------------
# Numerical microlocal diagnostics on the registry
# ---------------------------------------------------------------------------
def analyze_registry(reg, x_grid, xi_grid, y_grid=None, eta_grid=None,
                     ellipticity_threshold=1e-6):
    report = {}
    for name, op in reg.items():
        entry = {'order': None, 'homogeneous': None}
        try:
            if isinstance(op, MatrixPseudoDifferentialOperator):
                scalar_op = op.entries[0][0]
            else:
                scalar_op = op

            # --- FIX: use principal symbol for order detection ---
            # The full symbol is non-homogeneous (principal + subprincipal),
            # so symbol_order() on it gives wrong results.
            # Instead, check homogeneity of the principal symbol directly.
            principal = scalar_op.principal_symbol(order=1)
            temp_op = PseudoDifferentialOperator(
                principal, scalar_op.vars_x, mode='symbol')
            is_hom, deg = temp_op.is_homogeneous()
            if is_hom:
                entry['order'] = float(deg)
                entry['homogeneous'] = (True, deg)
            else:
                # Fallback: try symbol_order on principal only
                entry['order'] = temp_op.symbol_order()
                entry['homogeneous'] = (False, None)
        except Exception as e:
            entry['order'] = f'error: {e}'
            entry['homogeneous'] = None

        report[name] = entry

    # ellipticity check (unchanged)
    if 'laplace_beltrami' in reg:
        op = reg['laplace_beltrami']
        try:
            if op.dim == 1:
                ell = op.is_elliptic_numerically(
                    x_grid, xi_grid, threshold=ellipticity_threshold)
            else:
                ell = op.is_elliptic_numerically(
                    (x_grid, y_grid), (xi_grid, eta_grid),
                    threshold=ellipticity_threshold)
            report['laplace_beltrami']['elliptic'] = ell
        except Exception as e:
            report['laplace_beltrami']['elliptic'] = f'error: {e}'

    print('\n=== ψDO diagnostics ===')
    for name, e in report.items():
        extra = f'  elliptic={e["elliptic"]}' if 'elliptic' in e else ''
        print(f'  {name:28s} order={e["order"]}  '
              f'homogeneous={e["homogeneous"]}{extra}')
    return report


# ---------------------------------------------------------------------------
# BEYOND ψDOs: Bilinear Operators & Fourier Integral Operators (FIOs)
# ---------------------------------------------------------------------------
def print_microlocal_extensions(m):
    """
    Print the operations that fall OUTSIDE the standard linear ψDO registry,
    explaining their microlocal nature.
    """
    print("\n" + "="*78)
    print("BEYOND LINEAR ψDOs: Bilinear Operators & FIOs")
    print("="*78)
    
    x, y = m.coords
    xi, eta = _freq(m)
    
    print("\n1. BILINEAR OPERATORS (Algebraic / Order 0)")
    print("   These take TWO sections as input, so they are not linear ψDOs.")
    print("   • Inner Product <U, V>_g : Symbol is the inverse metric g^ij.")
    print("   • Outer Product U ⊗ V    : Symbol is the tensor product.")
    print("   • Lie Bracket [X, Y]     : Symbol is the Lie algebra structure.")
    
    print("\n2. PULLBACKS (Fourier Integral Operators - FIOs)")
    print("   Given a diffeomorphism φ: M → N, the pullback φ* maps Ω^k(N) → Ω^k(M).")
    print("   φ* is NOT a ψDO because it changes the underlying manifold.")
    print("   Instead, it is an FIO associated with the canonical transformation")
    print("   (the cotangent lift) of φ on the symplectic manifold T*M.")
    print("   • Symbolically: φ*(f)(x) = f(φ(x))")
    print("   • Microlocally: φ* propagates singularities along the bicharacteristics")
    print("     pushed forward by the cotangent lift of φ.")
    
    # Demonstrate the Pullback Symbolically using the new Metric method
    print("\n--- Symbolic Pullback Example ---")
    u, v = symbols('u v', real=True, positive=True)
    # Map from (u,v) to (x,y)
    phi_map = (u * cos(v), u * sin(v)) 
    omega_xy = (y, -x) # A 1-form in (x,y)
    
    omega_uv = m.pullback_1form(phi_map, omega_xy, (u, v))
    print(f"   Map φ: (u,v) ↦ (u cos v, u sin v)")
    print(f"   Form ω = y dx - x dy")
    print(f"   Pullback φ*ω = {omega_uv[0]} du + {omega_uv[1]} dv")

# Run the extensions
print_microlocal_extensions(m)

# Demonstrate the Lie Derivative Symbol with a specific vector field (e.g., Rotation)
# On the flat metric, rotation X = (-y, x) is a Killing field.
X_rot = (-u, v)
print_symbol_table(m, X_field=X_rot)


# 2. Real operators
reg = metric_symbol_registry(m)
print('\nRegistry operators:', list(reg.keys()))

# 3. Numerical diagnostics
ug  = np.linspace(-2, 2, 64)
vg  = np.linspace(0, 2*np.pi, 64)
xig = np.linspace(-8, 8, 64)
etg = np.linspace(-8, 8, 64)
report = analyze_registry(reg, ug, xig, y_grid=vg, eta_grid=etg)